# Breakout 2B · Trade and the border: mirrors, segments, networks, regimes, documents

Workshop **Data Driven Domestic Revenue Mobilization**, Bangkok, October 2026.

Mirror data read two ways and as a series, neighbours' prices on two rulers, importers as segments,
brokers as a scorecard, the network as a picture, the three regimes as totals, the documents behind a
declaration, and what similar goods were accepted at. The exercise is the 24 mirror gaps of the
worksheet, Country A's chapters on the regional map, and the five invoice lines of track B.

**How to use it.** Press ▶ on each cell from top to bottom, or *Runtime › Run all*. Change only the sliders and drop-down menus. The code is folded: double-click a title to see it. Every result ends with a **Finding** sentence.

Everything here is synthetic; Country A to J are stylised neighbours, and the partners carry real ISO
codes none of which is a participating country.

In [ ]:
#@title 0 · Where the data comes from
#@markdown Leave these as they are. Your analysts can point DATA_URL at their own extract (section 6).
DATA_URL = "https://raw.githubusercontent.com/FrancoisChastel/CustomsWorkshopNotebooks/data-v7/data"  #@param {type:"string"}
INSTALL = True  #@param {type:"boolean"}

In [ ]:
#@title 0 · Setup (run me first)
import sys, subprocess, warnings
warnings.filterwarnings("ignore")
if INSTALL and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/FrancoisChastel/CustomsWorkshopNotebooks@data-v7"], check=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from workshop_methods import notebook as nb
from workshop_methods.data import load_json, load_tables
from workshop_methods import border, trade, valuation
nb.use_style()
pd.set_option("display.max_colwidth", 40)
CALIBRATION = "random draw finds something on 1 to 8 lines in a hundred; this set is enriched on the 20 exercise lines only."
def calibration_line():
    """Printed under every table that shows a rate."""
    display(Markdown(f"<small><i>Calibration: {CALIBRATION}</i></small>"))
HOME_USE = ("home_use", "warehouse_out")  # the regimes where duty is paid: the lines a price check ranks
T = load_tables(DATA_URL, ['mirror_trade', 'regional_unit_values', 'sector_regional', 'trader_features', 'features_lines', 'warehouse_stock', 'transit', 'parcels', 'accepted_history', 'invoice_lines', 'valuation_neighbours', 'fx_rates', 'brokers', 'exporters'])
print(f"Loaded {len(T)} tables from {DATA_URL}. Ready.")

In [ ]:
#@title 0 · Prepare the tables
mirror, ruv, sector, tf, f = T["mirror_trade"], T["regional_unit_values"], T["sector_regional"], T["trader_features"], T["features_lines"]
rated = trade.mirror_ratios(mirror)
latest = int(mirror["year"].max())
print(f"Mirror data for {mirror['chapter'].nunique()} chapters x {mirror['partner'].nunique()} partners, {mirror['year'].min()} to {latest}; "
      f"regional prices for {ruv['commodity'].nunique()} goods, {ruv['importer'].nunique()} countries, {ruv['ruler'].nunique()} rulers. "
      f"{len(f):,} lines, {len(tf):,} importers, {len(T['brokers'])} brokers, {len(T['exporters'])} sellers.")

## 1 · Mirror data, two ways

The partner's export statistics are a second witness of what crossed the border. Our customs value
includes shipping and insurance (CIF); the partner's export figure does not (FOB). Where our side holds
the FOB value, freight and insurance as separate fields, compare **FOB to FOB**; where it does not,
take the **chapter wedge** (3% to 28%, median 6%) off the ratio. The table also says whether the
partner's weights are reported or estimated from value, whether it attributes trade to origin or to
consignment, whether it reported at all, and which trade system it uses.

In [ ]:
#@title 1 · One corridor, two ways
CHAPTER = "39"  #@param ["17","33","39","52","61","72","85","87"]
PARTNER = "CHN"  #@param ["CHN","MYS","IND"]
nb.steps("Does home record what the partner says it shipped?",
         "Divide home imports by the partner's exports, year by year: the raw ratio (our CIF over its FOB).",
         "Where the FOB field exists on our side, divide our FOB by the partner's FOB; where it does not, divide the raw ratio by 1 + the chapter wedge.",
         "Read the partner-side fields beside the numbers: reported weights, attribution, whether it reported, its trade system.")
s = rated[(rated["chapter"] == CHAPTER) & (rated["partner"] == PARTNER)].sort_values("year")
fig, ax = nb.chart(f"Chapter {CHAPTER} from {PARTNER}: home imports against the partner's exports")
ax.plot(s["year"], s["ratio"], marker="o", color=nb.COLORS["muted"], label="raw: our CIF over the partner's FOB")
ax.plot(s["year"], s["ratio_fob"], marker="o", color=nb.COLORS["accent"], label="FOB to FOB (the chapter wedge where no FOB field)")
ax.axhline(1.0, linestyle="--", color=nb.COLORS["ink"], label="an honest corridor sits near 1")
ax.set_xticks(s["year"]); ax.set_ylim(0, max(1.4, float(np.nanmax(s["ratio"])) * 1.1)); ax.legend(fontsize=8); plt.show()
fields = s[["year", "partner_exports_fob_usd", "home_imports_cif_usd", "home_imports_fob_usd", "reported_weight_flag", "attribution", "partner_reports", "trade_system"]]
nb.show(fields.set_index("year").tail(4), formats={c: "{:,.0f}" for c in ["partner_exports_fob_usd", "home_imports_cif_usd", "home_imports_fob_usd"]})
last = s.iloc[-1]
basis = "our FOB field" if pd.notna(last["home_imports_fob_usd"]) else f"the chapter wedge of {last['cif_fob_adj']:.0%}"
weights = "reported" if last["reported_weight_flag"] else "estimated from value"
if pd.isna(last["ratio"]) or not last["partner_reports"]:
    nb.finding(f"{PARTNER} did not report its {latest} exports in chapter {CHAPTER}: the mirror needs better data, not a query.")
else:
    nb.finding(f"In {latest} home recorded {last['ratio']:.0%} of what {PARTNER} reports shipping in chapter {CHAPTER} (raw); FOB to FOB "
               f"({basis}) it is {last['ratio_fob']:.0%}. The partner's weights are {weights} and it attributes trade by "
               f"{last['attribution']}: read these before reading the gap.")

In [ ]:
#@title 1 · The bridge: subtract each known explanation
CIF_FOB = 0.08  #@param {type:"slider", min:0, max:0.28, step:0.01}
TIMING = True  #@param {type:"boolean"}
HUB_SHARE = 1.0  #@param {type:"slider", min:0, max:1, step:0.25}
nb.steps("How much of the chapter 39 gap from the first partner survives every known explanation?",
         "This corridor has no FOB field on our side, so start from the partner's exports and add the chapter wedge: the expected home imports.",
         "Add back goods in transit at the year end (timing) and goods attributed to the hub that re-ships them.",
         "What is left is the residual worth a query.")
row = mirror[(mirror["chapter"] == "39") & (mirror["partner"] == "CHN") & (mirror["year"] == latest)].iloc[0]
b = trade.bridge(row["partner_exports_fob_usd"] / 1e6, row["home_imports_cif_usd"] / 1e6, CIF_FOB,
                 row["timing_usd"] / 1e6 if TIMING else 0.0, row["hub_attributed_usd"] / 1e6 * HUB_SHARE)
steps_ = {"partner exports": b["partner_exports"], "expected at home": b["expected_cif"], "recorded": b["recorded"],
          "+ timing": b["timing"], "+ hub": b["hub"], "residual": b["residual"]}
fig, ax = nb.chart(f"Chapter 39 from CHN, {latest}: {b['residual_share']:.0%} of expected imports unexplained")
colors = [nb.COLORS["muted"], nb.COLORS["navy2"], nb.COLORS["accent"], nb.COLORS["pale"], nb.COLORS["pale"], nb.COLORS["amber"]]
ax.bar(list(steps_), list(steps_.values()), color=colors)
for i, v in enumerate(steps_.values()):
    ax.text(i, v, f"{v:.1f}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("USD million"); plt.show()
nb.finding(f"Partner exports of USD {b['partner_exports']:.1f} million should arrive as {b['expected_cif']:.1f} million with shipping and "
           f"insurance at {CIF_FOB:.0%}; recorded {b['recorded']:.1f} + timing {b['timing']:.1f} + hub {b['hub']:.1f} explain {b['explained']:.1f}. "
           f"The residual, {b['residual']:.1f} million or {b['residual_share']:.0%} of expected, is a query to the partner administration, not a verdict.")

In [ ]:
#@title 1b · Regional prices: what neighbours pay for the same goods, on two rulers
RULER = "exporter_stats"  #@param ["exporter_stats", "importer_stats_fob"]
THRESHOLD = 0.70  #@param {type:"slider", min:0.5, max:0.9, step:0.05}
nb.steps("Does Country A pay much less than its neighbours for the same goods from the same seller?",
         "For each good and year, divide each country's value per kilo by the median of the ten importers, on the ruler chosen "
         "(the seller's export statistics, or the importers' own statistics on an FOB basis).",
         f"Flag Country A where it pays less than {THRESHOLD:.0%} of the median; trust the ratio only where reported weights cover most of the value.")
flags = trade.regional_flags(ruv, threshold=THRESHOLD, ruler=RULER)
part = ruv[ruv["ruler"] == RULER]
dots = part.groupby(["commodity", "importer"])["uv_ratio"].mean().unstack()
order = flags.sort_values("ratio", ascending=False)["commodity"].tolist()
fig, ax = nb.chart(f"Country A against the regional median ({RULER.replace('_', ' ')})")
for i, commodity in enumerate(order):
    others = dots.loc[commodity].drop("Country A")
    ax.scatter(others, [i] * len(others), color=nb.COLORS["muted"], s=18)
    a = dots.at[commodity, "Country A"]
    ax.scatter([a], [i], color=nb.COLORS["amber"] if a < THRESHOLD else nb.COLORS["accent"], s=70, zorder=3)
ax.axvline(1.0, color=nb.COLORS["ink"], linewidth=1); ax.axvline(THRESHOLD, color=nb.COLORS["amber"], linestyle=":")
ax.set_yticks(range(len(order)), [c.replace("_", " ") for c in order]); ax.set_xlabel("ratio to the regional median"); plt.show()
nb.show(flags.set_index("commodity")[["ratio", "weight_coverage", "years", "below_threshold", "trusted"]].rename(columns={"weight_coverage": "reported_weight_share"}),
        formats={"ratio": "{:.2f}", "reported_weight_share": "{:.0%}"}, highlight=flags.set_index("commodity")["below_threshold"])
low = flags[flags["below_threshold"]]
trusted, doubtful = low[low["trusted"]], low[~low["trusted"]]
text = ", ".join(f"{c.replace('_', ' ')} ({r:.2f})" for c, r in zip(trusted["commodity"], trusted["ratio"]))
extra = "".join(f" {c.replace('_', ' ')} ({r:.2f}) is low too, but reported weights cover only {w:.0%} of its value: a grade mix, not a price." for c, r, w in
                zip(doubtful["commodity"], doubtful["ratio"], doubtful["weight_coverage"]))
other = trade.regional_flags(ruv, threshold=THRESHOLD, ruler=[r for r in ruv["ruler"].unique() if r != RULER][0])
same = set(trusted["commodity"]) == set(other[other["below_threshold"] & other["trusted"]]["commodity"])
nb.finding(f"Country A pays under {THRESHOLD:.0%} of the regional median for {text or 'no good'}, all three years.{extra} "
           + ("The other ruler flags the same goods: a gap that holds on both rulers is a national valuation question." if same
              else "The other ruler flags a different set: read both before raising it in the regional working group."))

In [ ]:
#@title 1c · Risky sectors: the chapter by country map
YEAR = 2025  #@param [2023, 2024, 2025]
nb.steps("Where in the region are prices low, by chapter?",
         "Value-weighted ratio to the regional median, chapter by country; cells where reported weights cover under 70% of the value are greyed out.",
         "Read a column: one country's audit map. Read a row: a sector worth a regional conversation.")
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
grid = trade.sector_map(sector, YEAR)
cmap = LinearSegmentedColormap.from_list("ws", [nb.COLORS["amber"], "#FFFFFF", nb.COLORS["accent"]])
cmap.set_bad(nb.COLORS["border"])
fig, ax = nb.chart(f"{YEAR}: chapters below the regional median are amber; grey cells rest on estimated weights", height=4.8)
ax.imshow(np.ma.masked_invalid(grid.to_numpy()), cmap=cmap, norm=TwoSlopeNorm(vcenter=1.0, vmin=0.5, vmax=1.5), aspect="auto")
ax.set_xticks(range(grid.shape[1]), [c.replace("Country ", "") for c in grid.columns])
ax.set_yticks(range(grid.shape[0]), [f"ch. {c}" for c in grid.index]); ax.grid(False)
for (i, j), v in np.ndenumerate(grid.to_numpy()):
    if pd.notna(v) and v < 0.70:
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color=nb.COLORS["navy"])
ax.set_xlabel("country (A to J)"); plt.show()
column = grid["Country A"]
low_a = column[column < 0.70]
greyed = int(column.isna().sum())
elsewhere = grid.drop(columns="Country A").stack()
elsewhere = elsewhere[elsewhere < 0.70]
listed = [f"{c} ({v:.2f})" for c, v in low_a.items()]
listed = ", ".join(listed[:-1]) + " and " + listed[-1] if len(listed) > 1 else "".join(listed) or "none"
nb.finding(f"Country A's column is low in chapters {listed}; {greyed} of its chapters are greyed (weights cover under 70% of the value, so the "
           f"ratio says little). {len(elsewhere)} other cells are below 0.70 ({', '.join(f'ch. {c} in {k}' for (c, k) in elsewhere.index[:4])}): "
           "a row with several low cells is a sector for the regional conversation, a column is one country's audit plan.")

## 2 · Reading a series

In [ ]:
#@title 2 · Persistent: every year below 0.80 · growing: the last two years below · spike: one year 30 points below the rest · new: no history
nb.steps("What shape does each corridor's gap have over the eight years?",
         "Use the ratio after shipping, timing and hubs, year by year.",
         "Persistent: every year below 0.80. Growing: the last two years below 0.80 after years above it. Spike: one year at least 30 points below the others. "
         "New: the corridor appears in the last two years. Normal: the rest.")
SHAPES = ["persistent", "growing", "spike", "new"]
shapes = trade.classify_series(mirror).replace([np.inf, -np.inf], np.nan)
flagged = shapes[shapes["shape"].isin(SHAPES)]
examples = flagged.drop_duplicates("shape").set_index("shape").loc[[s for s in SHAPES if s in set(flagged["shape"])]]
fig, axes = plt.subplots(1, len(examples), figsize=(8.6, 3.2), sharey=True)
fig.suptitle("One corridor of each shape", x=0.01, ha="left", color=nb.COLORS["navy"], fontweight="bold")
for ax, (shape, ex) in zip(np.atleast_1d(axes), examples.iterrows()):
    s = rated[(rated["chapter"] == ex["chapter"]) & (rated["partner"] == ex["partner"])].sort_values("year")
    ax.plot(s["year"], s["ratio_after_known"].replace([np.inf, -np.inf], np.nan), marker="o", color=nb.COLORS["amber"])
    ax.axhline(0.80, color=nb.COLORS["muted"], linestyle=":"); ax.set_title(f"{shape}: ch. {ex['chapter']}, {ex['partner']}", fontsize=10)
    ax.set_xticks(s["year"][::2]); ax.set_ylim(0, 1.6)
plt.show()
nb.show(flagged[["chapter", "partner", "shape", "mean_ratio", "years_reported"]].reset_index(drop=True), formats={"mean_ratio": "{:.2f}"})
words = {s: ", ".join(f"ch. {c} {p}" for c, p in zip(g["chapter"], g["partner"])) for s, g in flagged.groupby("shape")}
unreported = shapes[shapes["years_reported"] == 0]
nb.finding("; ".join(f"{len(flagged[flagged['shape'] == s])} {s} ({words[s]})" for s in SHAPES if s in words)
           + f". {len(unreported)} corridor was never reported by the partner (ch. {unreported['chapter'].iloc[0]} {unreported['partner'].iloc[0]}: better data, "
           f"not a query), and the other {len(shapes) - len(flagged) - len(unreported)} look normal once the known explanations are applied. "
           "A persistent or growing gap is a practice; a spike is an event; a new corridor has no history yet.")

## 3 · Segments and brokers

In [ ]:
#@title 3 · Four segments of importers, and the control record of each
K = 4  #@param {type:"slider", min:2, max:6, step:1}
RERUN = False  #@param {type:"boolean"}
nb.steps("Which groups of importers deserve a closer look, and does their record agree?",
         "Every importer carries a precomputed segment (k-means with k = 4 on six trader signals: hit rate where controlled, learned score, "
         "price against peers, size, a blank tax ID, weight deviation).",
         f"Tick 'rerun' to recompute the groups here with k = {K} and see which groups survive.",
         "Read each segment's hit rate on months 1-18, where its importers were controlled.")
features = load_json(DATA_URL, "cluster_centres")["features"]
seg = border.segments(tf, features, k=K)["segment"] if RERUN else tf["importer_cluster"]
per = tf.assign(segment=seg).groupby("segment").agg(importers=("importer_id", "size"), median_value_12m=("value_12m_lcu", "median"),
                                                     hit_rate_h1=("hit_rate_h1", "mean"), blank_tax_id=("tin_blank", "mean"),
                                                     mean_learned_score=("mean_learned_score", "mean"), price_vs_peers=("median_share_of_median", "median"))
nb.show(per, formats={"median_value_12m": "{:,.0f}", "hit_rate_h1": "{:.0%}", "blank_tax_id": "{:.0%}", "mean_learned_score": "{:.2f}", "price_vs_peers": "{:.2f}"})
calibration_line()
fig, ax = nb.chart("Importers by size and price against peers, coloured by segment" + (" (recomputed)" if RERUN else " (precomputed, k = 4)"))
palette = [nb.COLORS["muted"], nb.COLORS["accent"], nb.COLORS["pale"], nb.COLORS["amber"], nb.COLORS["navy2"], nb.COLORS["soft"]]
for s, part in tf.assign(segment=seg).groupby("segment"):
    ax.scatter(part["value_12m_lcu"].clip(lower=1e5) / 1e6, part["median_share_of_median"], s=14, color=palette[(int(s) - 1) % len(palette)], label=f"segment {s}")
ax.set_xscale("log"); ax.set_xlabel("imports, 12 months, LCU million (log scale)"); ax.set_ylabel("typical price against peers"); ax.legend(fontsize=8); plt.show()
riskiest = per["hit_rate_h1"].idxmax()
nb.finding(f"Segment {riskiest} holds {per.at[riskiest, 'importers']} importers with a hit rate of {per.at[riskiest, 'hit_rate_h1']:.0%} where "
           f"controlled, against {per['hit_rate_h1'].min():.0%} for the quietest segment; its typical price is {per.at[riskiest, 'price_vs_peers']:.2f} of "
           "peers. A treatment per segment (facilitate the quiet ones, audit the top one) is how a budget becomes a plan; "
           "change k and keep only the groups that survive.")

In [ ]:
#@title 3 · The broker scorecard
nb.steps("Which brokers serve many importers with findings?",
         "For each broker with five clients or more: clients, lines, the share of its clients with a finding in months 1-18 (precomputed),",
         "its amendment rate and its own hit rate where its lines were controlled, against the median broker.")
h1 = f[f["period_month"] <= 18]
g = f.groupby("broker_id")
card = pd.DataFrame({"clients": g["importer_id"].nunique(), "lines": g.size(), "flagged_client_share": g["broker_flagged_share_h1"].first(),
                     "amendment_rate": g["status"].apply(lambda s: (s == "amended").mean()),
                     "hit_rate_h1": h1.groupby("broker_id")["outcome"].apply(lambda s: (s.dropna() == "finding").mean())})
card = card[card["clients"] >= 5].sort_values("flagged_client_share", ascending=False)
card["vs_median_broker"] = card["flagged_client_share"] - card["flagged_client_share"].median()
nb.show(card.head(8), formats={"flagged_client_share": "{:.0%}", "amendment_rate": "{:.1%}", "hit_rate_h1": "{:.0%}", "vs_median_broker": "{:+.0%}"})
calibration_line()
top = card.iloc[0]
nb.finding(f"Broker {card.index[0]}: {top['flagged_client_share']:.0%} of its {top['clients']:.0f} clients had a finding in months 1-18, "
           f"{top['vs_median_broker']:+.0%} against the median broker, and its own lines carry a finding {top['hit_rate_h1']:.0%} of the time where "
           "controlled. One broker can lodge for many risky importers: a compliance meeting with the broker, never a verdict on one client.")

## 4 · Networks and regimes

In [ ]:
#@title 4 · The network: who sits in the middle?
nb.steps("Which sellers connect to the most importers?",
         "Draw importers, brokers, consignees, sellers, addresses, phones and directors as dots, and every shared field as a line (network.json, with each node's degree and community).",
         "Count each seller's links; the projector game is to find the hub before the table does.")
net = load_json(DATA_URL, "network")
G = border.build_graph(net)
kinds = pd.Series([n["kind"] for n in net["nodes"]]).value_counts()
top = border.hubs(G, kind="exporter", top=10)
colors = {"importer": nb.COLORS["accent"], "exporter": nb.COLORS["amber"], "broker": nb.COLORS["navy2"], "consignee": nb.COLORS["pale"],
          "address": nb.COLORS["muted"], "phone": nb.COLORS["soft"], "director": nb.COLORS["ink"]}
sub = G.subgraph(set(top["node"]) | {n for x in top["node"] for n in G.neighbors(x)})
try:
    from pyvis.network import Network
    from IPython.display import HTML
    view = Network(height="520px", width="100%", notebook=True, cdn_resources="remote")
    for n, data in sub.nodes(data=True):
        view.add_node(n, label=n, color=colors.get(data["kind"], nb.COLORS["muted"]), size=6 + 1.5 * sub.degree(n), title=data["kind"])
    for a, b2 in sub.edges():
        view.add_edge(a, b2, color=nb.COLORS["border"])
    display(HTML(view.generate_html()))
except ImportError:
    import networkx as nx
    fig, ax = nb.chart("Sellers and their importers")
    pos = nx.spring_layout(sub, seed=1)
    nx.draw_networkx(sub, pos, ax=ax, node_size=[10 + 4 * sub.degree(n) for n in sub], with_labels=False,
                     node_color=[colors.get(sub.nodes[n]["kind"], nb.COLORS["muted"]) for n in sub], edge_color=nb.COLORS["border"])
    plt.show()
nb.show(top)
median = np.median([G.degree(n) for n, k in G.nodes(data="kind") if k == "exporter"])
communities = len({n["community"] for n in net["nodes"]})
nb.finding(f"The picture holds {len(net['nodes']):,} nodes ({', '.join(f'{v} {k}s' for k, v in kinds.items())}) in {communities} communities, drawn "
           f"from {len(f):,} lines, {len(tf):,} importers and {len(T['brokers'])} brokers. Sellers {top['node'][0]} and "
           f"{top['node'][1]} connect to {top['degree'][0]} and {top['degree'][1]} importers, against a median of {median:.0f} for all sellers: "
           "a hub spreads one practice across many importers.")

In [ ]:
#@title 4 · Open one node
NODE = "EXP0327"  #@param ["EXP0327", "EXP0322", "EXP0216", "EXP0395", "EXP0086", "EXP0203", "EXP0147", "EXP0595", "EXP0433", "EXP0254"]

nb.steps("What does this seller's trade look like?", "Take the lines it sold.", "Group them by importer.")
lines = f[f["exporter_id"] == NODE]
by = lines.groupby("importer_id").agg(lines=("line_id", "size"), value_lcu=("customs_value_lcu", "sum"),
                                      origin=("origin", lambda s: s.mode().iloc[0]), preference=("preference_claim", lambda s: (s == "FTA").mean()),
                                      price_vs_peers=("share_of_median", "median"))
nb.show(by.sort_values("value_lcu", ascending=False).head(10), formats={"value_lcu": "{:,.0f}", "preference": "{:.0%}", "price_vs_peers": "{:.2f}"})
nb.finding(f"Seller {NODE} sold to {len(by)} importers on {len(lines):,} lines; {(lines['preference_claim'] == 'FTA').mean():.0%} of its lines claim a "
           f"preference, mostly as origin {lines['origin'].mode().iloc[0]}, at {lines['share_of_median'].median():.2f} of the peer median.")

In [ ]:
#@title 4 · Regimes: warehouses that do not balance
nb.steps("Does each customs warehouse's stock balance?",
         "Opening stock + goods entered - goods exited - closing stock declared, as a share of goods entered.",
         "Flag months where more than 8% of entries are missing.")
gaps = border.warehouse_gaps(T["warehouse_stock"])
per = gaps.groupby("operator_id").agg(months=("period", "size"), months_flagged=("flag", "sum"), median_gap=("gap_share", "median"))
per = per.sort_values("median_gap", ascending=False)
nb.show(per, formats={"median_gap": "{:.1%}"}, highlight=per["months_flagged"] > per["months"] / 2)
leaking = per[per["months_flagged"] > per["months"] / 2]
lo, hi = leaking["median_gap"].min(), leaking["median_gap"].max()
spread = f"{lo:.0%}" if round(lo, 2) == round(hi, 2) else f"{lo:.0%} to {hi:.0%}"
nb.finding(f"{len(leaking)} operators lose more than 8% of entries in most months ({', '.join(leaking.index)}, a median of {spread} of entries); "
           "the others balance. Goods leaving a warehouse without a declaration pay no duty: stock-take the two.")

In [ ]:
#@title 4 · Regimes: transit that does not close
nb.steps("Do transit movements close, and on time?",
         "For each operator: the share of movements closed, hours against the route norm, movements open past the deadline.")
perf = border.transit_performance(T["transit"])
perf = perf[perf["movements"] >= 10]
nb.show(perf.head(8), formats={"closure_rate": "{:.0%}", "median_hours_vs_norm": "{:.2f}x"})
worst = perf.iloc[0]
nb.finding(f"Operator {perf.index[0]} closes only {worst['closure_rate']:.0%} of its {worst['movements']:.0f} movements, takes "
           f"{worst['median_hours_vs_norm']:.1f} times the route norm and leaves {worst['open_past_deadline']:.0f} open past the deadline; the median "
           f"operator closes {perf['closure_rate'].median():.0%}. An unclosed transit is goods released without duty: call the guarantees.")

In [ ]:
#@title 4 · Regimes: parcels just under the duty-free limit
DE_MINIMIS = 2000  #@param {type:"integer"}
nb.steps("Does anyone split shipments into parcels just under the duty-free limit?",
         f"For each receiver and month: parcels received and the largest parcel's value against {DE_MINIMIS:,} LCU.",
         "Flag months with six parcels or more, the largest above 90% of the limit.")
split = border.parcel_splitting(T["parcels"], de_minimis=DE_MINIMIS)
nb.show(split.head(8), formats={"max_value_lcu": "{:,.0f}"})
repeat = split[split["months_flagged"] >= 3]
nb.finding(f"{len(repeat)} receivers get six parcels or more a month, each just under the limit, in at least three months "
           f"({repeat['mean_consignments'].mean():.0f} parcels a month on average): one shipment split to stay duty free. "
           "Aggregate parcels by receiver before applying the limit.")

## 4b · Documents: what the papers say that the declaration does not

A declaration is a summary written by the declarant. Behind it sit documents written by other
hands, and each says something the declaration cannot.

In [ ]:
#@title 4b · The corpus behind one declaration, and the three conditions that turn it into data
corpus = pd.DataFrame({
    "written by": ["the seller", "the packer", "the carrier", "an issuing authority", "the scanner"],
    "what it says that the declaration does not": [
        "the price per unit, the currency, the terms of sale, discounts, the buyer and seller as they name themselves",
        "the number of packages, the weight per package, the marks and the model numbers",
        "the route, the ports, the dates, the shipper and the consignee as the carrier knows them, the container number",
        "who vouched for the origin, the standard or the licence, and when",
        "the shape and density of what is in the container, against what the packing list says"],
    "the check it allows": ["price against the invoice; the invoice against the accepted range (track B)", "weight against quantity; packages against the manifest",
                            "consignment against origin; dates against timing", "certificate against the issuing authority's register",
                            "image against the declared goods and their packing"],
}, index=["invoice", "packing list", "transport document", "certificates", "scanner image"])
nb.show(corpus)
conditions = pd.DataFrame({
    "condition": ["the document is filed with the declaration, as a file, not a paper copy",
                  "its fields are read into named columns (by a reader, a template, or a person), with the line they belong to",
                  "the reading is checked against the declaration and the difference is written back as a field an officer sees"],
    "without it": ["nothing can be read at scale", "a PDF is a picture, not data", "a difference nobody records never becomes a signal"],
}, index=["1", "2", "3"])
nb.show(conditions)
nb.finding("Five documents, each from a different hand, each allowing one check the declaration alone cannot; they become data only under the "
           "three conditions, and the third is the one administrations skip. Images are in production; reading documents is the newer step. "
           "(No synthetic invoice set is published with this release, so the table stands alone here.)")

## 1d · Valuation (track B): what similar goods were accepted at

For one invoice line, the accepted lines of the same heading and origin whose descriptions are most
alike (the similarities were computed once with a multilingual sentence model and stored in
`valuation_neighbours`), with every price put on one basis (US dollars, FOB, net of discounts). The
range is a reference for a question, never a value.

In [ ]:
#@title 1d · The neighbours of one invoice line
HINTS = ["Which fields make two descriptions similar?", "How many neighbours is enough?", "The range is never a value."]

LINE = "V2"  #@param ["V1", "V2", "V3", "V4", "V5"]
FLOOR = 0.85  #@param {type:"slider", min:0.7, max:0.95, step:0.01}
WINDOW_DAYS = 90  #@param {type:"slider", min:30, max:90, step:15}
K = 30  #@param {type:"slider", min:10, max:50, step:5}
HINT_LEVEL = 0  #@param {type:"slider", min:0, max:3, step:1}
history, invoice, neighbours = T["accepted_history"], T["invoice_lines"], T["valuation_neighbours"]
line = invoice.set_index("ref").loc[LINE]
nb.steps(f"What were goods like '{line['description']}' from {line['origin']} accepted at?",
         f"Keep accepted lines of the same heading ({line['hs6']}) and origin from the last {WINDOW_DAYS} days.",
         f"Keep those whose description is at least {FLOOR:.2f} alike, the {K} closest.",
         "Convert every price to US dollars, FOB, net of discounts; read the median and the middle half, and count the neighbours kept.",
         "Place the declared price: share of the median, spreads from it, below the middle half or not; decide nothing, an indicator, or a question.")
similarity = neighbours[neighbours["query_id"] == LINE].set_index("line_id")["similarity"].to_dict()
kept = valuation.neighbours(line, history, similarity, T["fx_rates"], k=K, floor=FLOOR, window_days=WINDOW_DAYS)
rng = valuation.price_range(kept)
declared = float(valuation.to_fob_usd(invoice[invoice["ref"] == LINE], T["fx_rates"], "invoice_date").iloc[0])
verdict = valuation.assess(declared, rng)
nb.show(kept[["accepted_on", "description", "currency", "incoterm", "unit_price", "fob_usd", "similarity"]].head(12),
        formats={"unit_price": "{:,.2f}", "fob_usd": "{:,.2f}", "similarity": "{:.2f}"})
nb.hint(HINT_LEVEL, HINTS, "2b.valuation.neighbours")
if rng["count"]:
    fig, ax = nb.chart(f"Line {LINE}: declared {declared:,.2f} against {rng['count']} neighbours")
    ax.hist(kept["fob_usd"], bins=12, color=nb.COLORS["pale"])
    ax.axvspan(rng["p25"], rng["p75"], color=nb.COLORS["accent"], alpha=0.15, label="middle half")
    ax.axvline(rng["median"], color=nb.COLORS["accent"], label=f"median {rng['median']:,.2f}")
    ax.axvline(declared, color=nb.COLORS["amber"], linewidth=2.5, label=f"declared {declared:,.2f}")
    ax.set_xlabel("unit price, USD FOB"); ax.legend(); plt.show()
    nb.finding(f"{rng['count']} neighbours above {FLOOR:.2f}, median {rng['median']:,.0f}, middle half {rng['p25']:,.0f} to {rng['p75']:,.0f}; "
               f"declared {declared:,.0f} = {verdict['share_of_median']:.0%} of the median, {verdict['distance']:.1f} spreads. "
               f"Decision: {verdict['decision']} ({verdict['reason']}). The range is never a value: the declared value can only be replaced through the valuation methods.")
else:
    nb.finding(f"No accepted line of the same goods and origin passes a floor of {FLOOR:.2f} within {WINDOW_DAYS} days: there is no range to read, and no question to ask from it.")

## 5 · Exercise: bin the 24 gaps, tick Country A's chapters, decide the five lines

Roles: the **analyst** reads the table, the **officer** bins each gap, the **committee** writes the first
check it would sign for every gap it wants investigated. Then tick the chapters where Country A pays more
than 30% below its neighbours with the weight coverage to trust it, and decide the five invoice lines of
track B. Nothing about the answer is visible until the reveal.

In [ ]:
#@title 5 · The 24 gaps
HINTS = ["Most gaps have a known explanation.", "Apply shipping and insurance, timing and hubs, then read the notes and weight flags before binning.", "Investigate only the pairs below 0.80 after shipping, timing and hubs that no note explains; tick a regional chapter only when Country A is below the threshold with the weight coverage to trust."]

HINT_LEVEL = 0  #@param {type:"slider", min:0, max:3, step:1}
ws = trade.mirror_ratios(pd.DataFrame(load_json(DATA_URL, "worksheet_2b")["rows"]))
ws["previous_ratio"] = (ws["home_imports_prev_usd"] / (ws["partner_exports_prev_usd"] * (1 + ws["cif_fob_adj"]))).round(3)
ws["gap"] = ws["chapter"] + " · " + ws["partner"]
ws["weights"] = np.where(ws["reported_weight_flag"] == 1, "reported", "estimated")
table = ws.set_index("gap")[["partner_exports_fob_usd", "home_imports_cif_usd", "ratio_fob", "ratio_after_known", "previous_ratio", "weights", "note"]]
nb.show(table, formats={"partner_exports_fob_usd": "{:,.0f}", "home_imports_cif_usd": "{:,.0f}", "ratio_fob": "{:.2f}",
                        "ratio_after_known": "{:.2f}", "previous_ratio": "{:.2f}"}, max_rows=24)
nb.hint(HINT_LEVEL, HINTS, "2b.exercise")
low = table[(table["ratio_after_known"] < 0.80) | (table["previous_ratio"] < 0.80)]
noted = int((table["note"] != "").sum())
nb.finding(f"{len(low)} of the 24 gaps sit below 0.80 this year or last after shipping, timing and hubs; {noted} rows carry a note. "
           "Read the notes and the weight flags before binning: most gaps have a known explanation.")

In [ ]:
#@title 5 · Bin each gap, tick Country A's chapters, lock
import ipywidgets as widgets
BINS, CHECKS = ["", "explained", "investigate", "needs better data"], ["", "ask the partner for its declarations", "check timing (shipped vs cleared)", "check hub re-exports", "check the weights", "check recoding between codes or headings", "audit the importers"]
regional = pd.DataFrame(load_json(DATA_URL, "worksheet_2b_regional")["cells"])
grid = regional.pivot(index="chapter", columns="importer", values="uv_ratio_to_region")
summary = pd.DataFrame({"Country A": grid["Country A"], "regional median": grid.drop(columns="Country A").median(axis=1),
                        "lowest neighbour": grid.drop(columns="Country A").min(axis=1),
                        "reported weight share, A": regional[regional["importer"] == "Country A"].set_index("chapter")["reported_weight_share"]})
nb.show(summary, formats={c: "{:.2f}" for c in summary.columns})
rows = [widgets.HBox([widgets.Label(g, layout=widgets.Layout(width="130px")),
                      widgets.Dropdown(options=BINS, layout=widgets.Layout(width="170px")),
                      widgets.Dropdown(options=CHECKS, layout=widgets.Layout(width="290px"))]) for g in table.index]
ticks = [widgets.Checkbox(description=f"chapter {c}", indent=False) for c in summary.index]
lock = widgets.Button(description="Lock my bins and ticks", button_style="primary")
note = widgets.Output()
LOCKED = {"bins": {}, "checks": {}, "chapters": []}
def on_lock(_):
    LOCKED["bins"] = {r.children[0].value: r.children[1].value for r in rows}
    LOCKED["checks"] = {r.children[0].value: r.children[2].value for r in rows}
    LOCKED["chapters"] = [t.description.split()[-1] for t in ticks if t.value]
    with note:
        note.clear_output()
        binned = sum(1 for v in LOCKED["bins"].values() if v)
        print(f"Locked: {binned} of 24 gaps binned, chapters ticked: {', '.join(LOCKED['chapters']) or 'none'}.")
lock.on_click(on_lock)
display(widgets.VBox(rows + [widgets.HBox(ticks), lock, note]))
#@markdown If the menus do not show, type the gaps you would investigate (e.g. "17 · MYS, 39 · CHN") and the chapters:
MY_INVESTIGATE = ""  #@param {type:"string"}
MY_CHAPTERS = ""  #@param {type:"string"}
if MY_INVESTIGATE or MY_CHAPTERS:
    LOCKED["bins"] = {g.strip(): "investigate" for g in MY_INVESTIGATE.split(",") if g.strip()}
    LOCKED["chapters"] = [c.strip() for c in MY_CHAPTERS.split(",") if c.strip()]
    print(f"Typed in: investigate {', '.join(LOCKED['bins'])}; chapters {', '.join(LOCKED['chapters'])}.")

In [ ]:
#@title 5 · Track B: the five invoice lines
import ipywidgets as widgets
DECISIONS = ["", "nothing", "indicator", "question"]
invoice = T["invoice_lines"]
rows_b = [widgets.HBox([widgets.Label(f"{r.ref} {r.description[:38]}", layout=widgets.Layout(width="330px")),
                        widgets.Dropdown(options=DECISIONS, layout=widgets.Layout(width="130px")),
                        widgets.Text(placeholder="the reason, in one line", layout=widgets.Layout(width="320px"))])
          for r in invoice.itertuples()]
lock_b = widgets.Button(description="Lock track B", button_style="primary")
note_b = widgets.Output()
LOCKED_B = {}
def on_lock_b(_):
    LOCKED_B.update({r.children[0].value.split()[0]: r.children[1].value for r in rows_b})
    with note_b:
        note_b.clear_output()
        print("Locked: " + ", ".join(f"{k} {v or '-'}" for k, v in LOCKED_B.items()))
lock_b.on_click(on_lock_b)
display(widgets.VBox(rows_b + [lock_b, note_b]))
#@markdown If the menus do not show, type "V1:nothing, V2:question, ..." here instead.
MY_DECISIONS = ""  #@param {type:"string"}
if MY_DECISIONS:
    LOCKED_B.update(dict(p.split(":", 1) for p in MY_DECISIONS.replace(" ", "").split(",") if ":" in p))
    print(f"Typed in: {LOCKED_B}")

## Reveal (facilitator)

The facilitator types the code. Without it, this cell prints nothing.

In [ ]:
#@title 5 · Reveal
REVEAL_CODE = ""  #@param {type:"string"}
SEALED = {"salt": "XpluhpYnwd8yRTdy8Fzpfg==", "data": "uAHQCjp2FmMVx7G6N9u4E3bFVMMFzpPT4SHy7F84HTmPNjF5Ga9BepSgsw05C5tTTNbB8DjE4N2frlJ5hAjKXUuQ7wJ2V3EyRPAHEmudURx44jsmnf3i/D2i3Nx5AwI3H0IAWKPib5jkQ3gLcZXIHJfmeoeSD+7YWMAmTjpWlp0C9e0YvyOp53AhteTOcbgsLTX7X7YJ/+EkuELIN9viTn00/OCmDd/uvAckADkjmtCqGxSX3hlRh3WJdI5lu4FvlW2kHZGCXJPYDuRjvUhi8mRWbr3hU0mHqGPeWzVve+rUXwf9XfCya3MIn18BbyneLRmUBANp08nIYzHR9a3dl+GsUAPBwcL1I+82J5rn5MEHSag4j9A03CqmYIE4imh5a+IG0hV06EaaLdSsGxdZnt61PxwmU+wWFNT31DT5elpCrr/8Uy9avDAbWCmOZLRTNvUdCOxmKzNhEOwrdw9GS/dpZ+BbE30qa0bb1ve+PGOpxnJfogflMAgx9KKXu6zWBuuWXIHrVrPMWFBKnbtB0eu01mIouUBY7MgqDyTZTT2r81DQ2MfFnZoKoji2pPbFddApp0669CxYiZN3J17ixakhT7hdkhFGiPq5oIDj7rHHhJ4Xp45HV+Wl/rQNEasu5MGqKY8LFL+DDQepmxwvG4jw78cQ70q9NiVXf52SPxa4r2pHs8YNPjzuJpFl/YHwnbCSpVZKXdHeFckNNVRv31J8zZQ9fdClbrDcZJkKB0v1XrW8Vz61Pdg+SAuUWdzqeWqwLXuinj6ht3OS8GVTVc0uceotkZ1coHXt5VtTO+TFtdeBq8gYeYKPtaNdUorqGyBJcuk3fUfS0swFIUAosvlGXI24kBA1VKcwVF8b2q6KDCGWJOub1tgvkzQGuLSYPJVhe+u61nrPlfGWuo4CrvRNb2n/gKCgi/c03cmoYBaq1DU06F4ZpA/3aLqNFL1cZNH73dkZM5A89apTd5XRCwI7MUvq8OiWpNyq5reepdl8sY8yrYoCN5wSIVcO4JXv0lnvWCJpQwbN3gFMuTBElGKAccIp36vCv1yIxFu4Se8qCBPxVNveqsMIOlc/KHpvMOimhY8tNvilWcEn4PymgbvpAcZklk6b+X2Lmx2K4JJO3zZokuvSwn2pYg/sYuWcnCPnZnDTj/35h6cxa9uQNp2KzKQm7EA2Klu4BVLXzpmpAMU89ShUF65Gp+EWBK0pE/BOdAUMMUVLyy8qdILz1AajKJVhg+of3+dZPDkxlwjgfV1RlAkOta09NpQ9GzHCIMk6uSubFJUfhQapA5re1umq+9vZnFA+gl8TDWcTNfZw9+kFIPTajLdb/0MBGE6mcFv2F5pMFAZNS2e8Je+pU8PeIFej1d+IqrT45JKFuXRtX5+RFLSfjhAf/EBihOkfiRk/dr7/g/d05SudtlifpvejtXDdfpxSRIpM4+TKt9bLNJ4Cijv23JewbMVH3esovlvrTlGlfD+hs1/utNY9a91t0TdgD5zCdSWhamBWYXYZscFOarPdRU28/wiQ4eD/dnEzri0nGCsnB0gwStYLYDfH6J2YfffjKbKi0emjlvNwAlkkFa7nGqOdj0BJ45lc6Ln4cXyIY5rk5dJTuVsoZWC/J3iRgxWx", "mac": "848810267d4365db740426d9711620beeb1ad29d8ace66663b9f307a8527ea35"}
answer = nb.unseal(REVEAL_CODE, SEALED)
if answer:
    key = pd.DataFrame(answer["rows"]).assign(gap=lambda x: x["chapter"] + " · " + x["partner"]).set_index("gap")
    key["your bin"] = key.index.map(lambda g: LOCKED["bins"].get(g, ""))
    nb.show(key[["bin", "shape", "your bin", "why"]], max_rows=24, highlight=key["bin"] == "investigate")
    seeded = key[key["bin"] == "investigate"]
    found = int((seeded["your bin"] == "investigate").sum())
    wasted = int(((key["bin"] != "investigate") & (key["your bin"] == "investigate")).sum())
    chapters = set(answer["regional"]["seeded"])
    ticked = set(LOCKED["chapters"])
    nb.finding(f"Your group found {found} of the {len(seeded)} gaps to investigate and sent {wasted} ordinary gaps to "
               f"investigation. Country A is low, with the weights to trust it, in chapter {', '.join(sorted(chapters))}; you ticked "
               f"{', '.join(sorted(ticked)) or 'none'}.")
    vb = pd.DataFrame(answer["valuation"]).set_index("ref")
    vb["your decision"] = vb.index.map(lambda r: LOCKED_B.get(r, "") if "LOCKED_B" in globals() else "")
    nb.show(vb[["neighbours", "share_of_median", "distance", "correct", "your decision"]], formats={"share_of_median": "{:.0%}", "distance": "{:.1f}"})
    for item in answer["explanations"]:
        print(f"- {item}")
    if nb.HINTS_USED:
        print("Hints used:", nb.HINTS_USED)

## 6 · For your analysts

**Mirror data from UN Comtrade.** Download by hand from the Comtrade website: choose the partner as
reporter, your country as partner, exports, HS 6-digit codes and the years, then export the CSV. Keep
the reporter's FOB values and net weights; flag rows whose weight is estimated from value (about 20% of
quantities are; Comtrade marks them). Map HS 6 to your 8-digit lines by the first six digits before
comparing. Compare FOB to FOB where your declarations hold FOB, freight and insurance as separate fields;
otherwise take the chapter wedge (OECD margins: 6.2% on average, 3.0% to 28.0% by chapter) off the ratio.
Note the partner's attribution (origin or consignment), whether it reported, and its trade system.

**Regional medians.** For each good and year, take the median value per kilo across the importing
countries, then each country's ratio to it, on each ruler (the seller's export statistics; the importers'
own statistics on an FOB basis). Trust a ratio only when reported weights cover most of the value (here
70%). The sector map is the same ratio, weighted by value, rolled up to the chapter, greyed under 70%.

**Series rules.** Persistent, spike, new and normal come from simple rules on the ratio after shipping,
timing and hubs: the level, the largest single-year drop, and whether the corridor has a history.

**Segments.** k-means puts every importer in a group even when the groups are not clear; change k and
keep only groups that survive. The broker signal is a share of a broker's clients with a finding: a
conversation with the broker, never a verdict on one client (82 of 99 administrations never measured
broker-assisted compliance).

**Networks.** Nodes are importers, brokers, consignees, sellers, addresses, phones and directors; an
edge is a shared field in the declarations or the register. Degree (links per node) finds hubs;
community detection finds groups.

**Valuation.** Neighbours by cosine similarity of multilingual sentence embeddings (a small CPU model,
run once; the similarities are stored), same heading and origin, floor 0.85, 90 days, k = 30; the range
is the median and the middle half, the distance is in spreads (median absolute deviation). WTO Valuation
Agreement, Decision 6.1, and the WCO valuation database guidelines: a difference is not by itself an
indicator of risk, and a value from the range is never the customs value.

**Border performance.** Time to release by lane and hit rate by lane are the two numbers to publish
monthly, beside the regime checks above.

In [ ]:
#@title 6 · The core computations
ratios = trade.mirror_ratios(mirror)                          # raw, FOB to FOB, after the chapter wedge, after timing and hubs
print(trade.classify_series(mirror).query("shape != 'normal'").to_string(index=False))
print(trade.regional_flags(ruv).to_string(index=False))
print(card.head(3).to_string())